In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold
import optuna
from sklearn.base import clone
from sklearn.metrics import average_precision_score

In [ ]:
import optuna.visualization as vis
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

In [ ]:
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv("../data/diabetes_binary_health_indicators_BRFSS2015.csv")
df.head()

In [ ]:
_target_cols = ['Diabetes_binary']
_binary_cols = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity',
                'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
_ordinal_cols = ['GenHlth', 'Education', 'Income', 'Age']
_gray_cols = ['MentHlth', 'PhysHlth']
_continuous_cols = ["BMI"]

In [ ]:
x = df.drop(_target_cols, axis=1)
y = df[_target_cols].to_numpy().ravel()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
POS_WEIGHT_RATIO = (y_train == 0).sum() / (y_train == 1).sum()
sampler_kwargs = dict(seed=RANDOM_STATE)

In [ ]:
def _cv_score_with_pruning(trial, pipeline, x, y, cv, fit_kwargs_fn=None) -> float:
    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(x, y)):
        x_tr, x_val = x.iloc[train_idx], x.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        fold_pipeline = clone(pipeline)
        fit_kwargs = fit_kwargs_fn(x_val, y_val) if fit_kwargs_fn is not None else {}
        fold_pipeline.fit(x_tr, y_tr, **fit_kwargs)

        y_prob = fold_pipeline.predict_proba(x_val)[:, 1]
        fold_scores.append(average_precision_score(y_val, y_prob))

        trial.report(np.mean(fold_scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

In [ ]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)


rf_pipeline = Pipeline([
    ('rf', rf_model)
])
rf_pipeline

In [ ]:
def rf_parameter_space(trial: optuna.Trial) -> dict:
    params = {
        "rf__n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "rf__max_depth": trial.suggest_int("max_depth", 4, 20),
        "rf__min_samples_split": trial.suggest_int("min_samples_split", 2, 500),
        "rf__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 150),
        "rf__max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "rf__max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
        "rf__class_weight": trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]),
        "rf__criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "rf__bootstrap": True,
    }
    return params

In [ ]:
def rf_objective(trial, x, y):
    space = rf_parameter_space(trial)
    space["rf__n_jobs"] = 1
    pipeline = clone(rf_pipeline).set_params(**space)
    return _cv_score_with_pruning(trial, pipeline, x, y, CV)

In [ ]:
rf_study = optuna.create_study(
    direction="maximize", study_name="rf_tuning",
    sampler=optuna.samplers.TPESampler(**sampler_kwargs),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
)
rf_study.optimize(lambda t: rf_objective(t, x_train, y_train), n_trials=40, n_jobs=1)

In [ ]:
print(rf_study.best_value, rf_study.best_params)

In [ ]:
def per_model_plots(study):
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_intermediate_values(study).show()
    vis.plot_slice(study).show()
    vis.plot_parallel_coordinate(study).show()
    vis.plot_contour(study).show()

In [ ]:
per_model_plots(rf_study)